# EDA · Fase 1: de los tres Excel al consumo diario de frío

**Qué hace este notebook.** Parte de los tres Excel de la planta y llega a una tabla `diario` con **una fila por día** y una columna `objetivo`: el consumo de frío (`Frio (Kw)`) del **día siguiente**, que es lo que el modelo va a predecir.

**Para quién es.** Para cualquiera del equipo que tenga que retomar el trabajo desde acá (preprocesamiento, modelado, informe) sin haber participado de este hito. Cada sección dice qué pregunta responde, muestra la evidencia y cierra con lo que se encontró y se decidió.

**Cómo leerlo.** De arriba abajo. El color de una tabla se usa solo cuando todas sus columnas miden lo mismo: ahí, el mismo valor tiene siempre el mismo color y cuanto más oscura la celda, mayor el valor (las escalas de conteos arrancan en 0). Si las columnas mezclan unidades, la tabla va sin color. En las diferencias con signo, azul es "de menos", rojo "de más" y gris es cero. El código reutilizable no está acá: vive en `src/auxiliar_functions.py` y el notebook lo importa.

**Requisitos.** Entorno `cervecera_env` activo y los tres Excel dentro de `data/` (no se versionan; ver README, sección "Datos"). Ejecutarlo completo tarda unos minutos, casi todo en leer los Excel.

## Índice

1. Preparación
2. Los tres archivos y sus hojas
3. Lectura de la hoja `Consolidado EE`
4. Cierre del día: 23:00 o 23:59
5. Diagnóstico de filas por fecha
6. Fechas con más de 25 filas
7. Los tres archivos juntos: períodos, solapes y diferencias
8. El calendario: fechas ausentes y bloques por mes
9. Evidencia de la hoja `Seguimiento Dia`
10. Decisiones sobre los bloques y los días inválidos
11. La tabla `diario`: una versión por bloque
12. El objetivo: el consumo del día siguiente
13. Pendientes

## Decisiones que salen de este notebook

| Código | Decisión | Dónde se justifica |
|---|---|---|
| **D1** | El consumo del día se toma de la última lectura (23:59), no de la de las 23:00. | Sección 4 |
| **D2** | Cada bloque de lecturas de 00:00 a 23:59 es un día distinto, aunque comparta fecha con otro. "Día siguiente" significa el bloque siguiente. | Secciones 6, 8, 9 y 10 |
| **D3** | Si un día está en dos Excel se usa la versión completa y plausible; si empatan, la del Excel más reciente. | Sección 7 |
| **D4** | Los días sin lectura de las 23:59 o con consumos del orden de millones son inválidos: se marcan y **no se imputan**. | Secciones 10 y 11 |

## 1. Preparación

Se importan las librerías y las funciones compartidas de `src/auxiliar_functions.py`. Las rutas son relativas a la carpeta `notebooks/`, así que el notebook se tiene que abrir desde ahí (VS Code y Jupyter lo hacen por defecto).

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

# Los scripts del TP importan como `from auxiliar_functions import ...`; acá se hace igual.
sys.path.append("../src")
from auxiliar_functions import (
    COLUMNA_CONSUMO,
    CONSUMO_MAXIMO_PLAUSIBLE,
    HORA_CIERRE,
    cierres_por_bloque,
    construir_diario,
    formatear_fechas,
    leer_consolidado_ee,
    mostrar_lado_a_lado,
    mostrar_lado_a_lado_simple,
    numerar_bloques,
    resumir_por_fecha,
    tabla_simple,
    unir_versiones,
)

DATA_DIR = Path("../data")
archivos = sorted(DATA_DIR.glob("Totalizadores Planta *.xlsx"))
assert len(archivos) == 3, "Faltan Excel en data/ (ver README, sección 'Datos')."

# Nombre corto de cada archivo, p. ej. "2020-2022". El orden alfabético es también el cronológico.
nombres = [archivo.stem.replace("Totalizadores Planta ", "") for archivo in archivos]
print("Archivos encontrados:", nombres)

Archivos encontrados: ['2020-2022', '2021-2023', '2022-2023']


## 2. Los tres archivos y sus hojas

**Pregunta:** ¿qué hojas tiene cada Excel? ¿Se pueden leer por posición?

Los tres archivos cubren períodos distintos que se solapan en el tiempo. Antes de leer nada hay que saber si todos tienen la misma estructura. Cada libro se abre una sola vez y se reutiliza después, porque abrir un Excel de 30 MB es lo lento.

In [2]:
libros = {nombre: pd.ExcelFile(archivo) for nombre, archivo in zip(nombres, archivos)}
hojas = {nombre: libro.sheet_names for nombre, libro in libros.items()}

# Todas las hojas que aparecen en algún archivo, en el orden en que aparecen.
orden_de_hojas = []
for lista in hojas.values():
    for hoja in lista:
        if hoja not in orden_de_hojas:
            orden_de_hojas.append(hoja)

cantidad = pd.DataFrame({"Cantidad de hojas": {nombre: len(lista) for nombre, lista in hojas.items()}})
presencia = pd.DataFrame(
    {nombre: [int(hoja in lista) for hoja in orden_de_hojas] for nombre, lista in hojas.items()},
    index=orden_de_hojas,
)

# La matriz de presencia usa una sola escala fija (0 a 1) para toda la tabla: así cada 1
# tiene el mismo color en cualquier columna. Con una escala por columna, una columna
# toda de unos quedaba sin color.
mostrar_lado_a_lado(
    [
        ("Cantidad de hojas por archivo", cantidad, {"vmin": 0}),
        ("Hojas presentes (1 = está, 0 = no está)", presencia, {"axis": None, "vmin": 0, "vmax": 1}),
    ],
    precision=0,
)

**Hallazgo.** Hay 15 hojas comunes a los tres archivos. El segundo trae una más, `Seguimiento Dia`, que los otros no tienen; por eso las cantidades son 15, 16 y 15. Las hojas se leen siempre **por nombre** y no por posición. En este notebook solo se usa `Consolidado EE`; el inventario del resto queda como pendiente (sección 13).

## 3. Lectura de la hoja `Consolidado EE`

**Pregunta:** ¿cómo es la hoja que contiene el objetivo y cómo se ve un día?

El objetivo, `Frio (Kw)`, está en la hoja `Consolidado EE`. La función `leer_consolidado_ee` la busca **por nombre** (no por posición, porque los archivos tienen distinta cantidad de hojas), descarta las filas vacías del final y convierte `DIA` a fecha. Es el paso lento: alrededor de un minuto por archivo.

In [3]:
tablas = {}
filas_sin_fecha = {}
for nombre, libro in libros.items():
    tablas[nombre], filas_sin_fecha[nombre] = leer_consolidado_ee(libro)

tabla_a = tablas[nombres[0]]  # el primer archivo se usa como ejemplo en las secciones 3 a 6

dimensiones = pd.DataFrame({"Cantidad": {"Filas con fecha": len(tabla_a), "Columnas": tabla_a.shape[1]}})
tipos = pd.DataFrame({"Tipo de dato": tabla_a.dtypes.astype(str)})

mostrar_lado_a_lado_simple(
    [(f"Dimensiones de la hoja (archivo {nombres[0]})", dimensiones), ("Columnas y tipo de dato", tipos)],
    precision=0,
)

,Cantidad
Filas con fecha,15575
Columnas,21
,Tipo de dato
DIA,datetime64[us]
HORA,str
Planta (Kw),float64
Elaboracion (Kw),float64
Bodega (Kw),float64
Cocina (Kw),float64
Envasado (Kw),float64


Un día de ejemplo, el 01/07/2020, con cinco lecturas y la primera del día siguiente. Se muestran solo cuatro de las 19 columnas numéricas.

In [4]:
horas_ejemplo = ["00:00:00", "04:00:00", "12:00:00", "23:00:00", "23:59:00"]
es_dia_ejemplo = (tabla_a["DIA"] == "2020-07-01") & tabla_a["HORA"].isin(horas_ejemplo)
es_inicio_del_siguiente = (tabla_a["DIA"] == "2020-07-02") & (tabla_a["HORA"] == "00:00:00")

ejemplo = tabla_a.loc[es_dia_ejemplo | es_inicio_del_siguiente, ["DIA", "HORA", "Planta (Kw)", "Sala Maq (Kw)", COLUMNA_CONSUMO, "KW Gral Planta"]]
ejemplo.index.name = "Fila del archivo"
display(tabla_simple(formatear_fechas(ejemplo), precision=0))

,DIA,HORA,Planta (Kw),Sala Maq (Kw),Frio (Kw),KW Gral Planta
Fila del archivo,,,,,,
0,2020-07-01,00:00:00,0,0,0,0
4,2020-07-01,04:00:00,2804,1701,1480,2978
12,2020-07-01,12:00:00,13387,8133,6872,14219
23,2020-07-01,23:00:00,25926,16304,13694,27716
24,2020-07-01,23:59:00,27637,17080,14325,29529
25,2020-07-02,00:00:00,0,0,0,0


**Hallazgo.** El primer archivo tiene 15.575 filas con fecha y 21 columnas: `DIA`, `HORA` y 19 variables numéricas en kW. Cada fila es una lectura horaria y los valores son **acumulados del día**: arrancan en 0 a las 00:00, crecen durante el día y se **reinician a medianoche**. Un día normal tiene 25 lecturas: 24 en punto (de 00:00 a 23:00) y una de cierre a las 23:59. El consumo del día es el acumulado de esa última lectura, no una suma de las horas.

## 4. Cierre del día: 23:00 o 23:59

**Pregunta:** ¿el consumo diario se toma de la lectura de las 23:00 o de la de las 23:59?

La consigna exige elegir una, aplicarla en todo el trabajo y justificarla. **Decisión D1: 23:59.** Lo que se quiere predecir es el consumo de un **día entero**; la lectura de las 23:00 deja afuera la última hora. Para ponerle número a lo que se pierde, se compara en cada día del primer archivo el acumulado de las 23:00 con el de las 23:59.

In [5]:
bloque_a = numerar_bloques(tabla_a)
es_23 = tabla_a["HORA"] == "23:00:00"
es_cierre = tabla_a["HORA"] >= HORA_CIERRE

# Un valor por bloque diario (la última lectura que cumple cada condición).
consumo_23 = tabla_a.loc[es_23, COLUMNA_CONSUMO].groupby(bloque_a[es_23]).last()
consumo_cierre = tabla_a.loc[es_cierre, COLUMNA_CONSUMO].groupby(bloque_a[es_cierre]).last()

comparacion_cierre = pd.DataFrame({"a las 23:00": consumo_23, "a las 23:59": consumo_cierre}).dropna()
# Se excluyen los errores del orden de millones: no dicen nada de la última hora.
comparacion_cierre = comparacion_cierre[comparacion_cierre["a las 23:59"] <= CONSUMO_MAXIMO_PLAUSIBLE]
comparacion_cierre["lo que se pierde (kW)"] = comparacion_cierre["a las 23:59"] - comparacion_cierre["a las 23:00"]
comparacion_cierre["% del total del día"] = 100 * comparacion_cierre["lo que se pierde (kW)"] / comparacion_cierre["a las 23:59"]

resumen_cierre = comparacion_cierre.describe(percentiles=[0.1, 0.5, 0.9]).loc[["mean", "10%", "50%", "90%"]]
resumen_cierre.index = ["promedio", "percentil 10", "mediana", "percentil 90"]

print("Días comparados:", len(comparacion_cierre))
# Sin color: las columnas mezclan kW y porcentajes, y un gradiente compartiría escala entre unidades distintas.
mostrar_lado_a_lado_simple([("Lo que se pierde al cortar el día a las 23:00 (primer archivo)", resumen_cierre)], precision=1)

Días comparados: 606


,a las 23:00,a las 23:59,lo que se pierde (kW),% del total del día
promedio,23005.2,24015.9,1010.7,4.2
percentil 10,11633.0,12105.5,470.5,2.9
mediana,23687.5,24825.5,960.5,4.0
percentil 90,32441.5,33782.5,1496.0,5.5


**Hallazgo.** Cortar a las 23:00 deja afuera, en promedio, **1.011 kW** de los 24.016 kW del día (**4,2 %**). La mediana es 961 kW (4,0 %) y en el 80 % central de los días va de 471 kW (2,9 %) a 1.496 kW (5,5 %). Se compararon 606 días del primer archivo. Es una hora entera de consumo, no un error de redondeo, y siempre en el mismo sentido: la lectura de las 23:00 **subestima** el día.

**D1: se usa la lectura de las 23:59.** Algunas lecturas de cierre llevan unos segundos de diferencia (23:59:01 o 23:59:59) y se tratan igual: por eso la condición del código es "hora mayor o igual a 23:59:00".

## 5. Diagnóstico de filas por fecha

**Pregunta:** ¿cuántas lecturas tiene cada fecha y a qué hora termina el día?

La consigna pide verificar que **ninguna fecha tenga más de 25 registros** antes de agregar, y detectar los días con menos de 24 lecturas. Se hace con el primer archivo; los otros dos se revisan en la sección 7.

In [6]:
resumen_a = resumir_por_fecha(tabla_a)

filas_por_fecha = resumen_a["filas"].value_counts().sort_index().rename_axis("Filas en la fecha").to_frame("Cantidad de fechas")
ultima_lectura = resumen_a["ultima_hora"].value_counts().rename_axis("Última lectura del día").to_frame("Cantidad de fechas")
vacias = pd.DataFrame({"Filas vacías al final de la hoja": filas_sin_fecha})

# Son conteos: la escala arranca en 0 para que 134 y 479 no parezcan "claro y oscuro" por estirarse entre sí.
desde_cero = {"vmin": 0}
mostrar_lado_a_lado(
    [
        (f"Filas por fecha (archivo {nombres[0]})", filas_por_fecha, desde_cero),
        ("Hora de la última lectura", ultima_lectura, desde_cero),
        ("Filas sin fecha descartadas, por archivo", vacias, desde_cero),
    ],
    precision=0,
)

,Cantidad de fechas
Filas en la fecha,
8,1
23,2
24,5
25,596
50,5
51,1
100,2
,Cantidad de fechas
Última lectura del día,


Las fechas "raras": las que no tienen 25 filas o cuya última lectura es anterior a las 23:59.

In [7]:
raras = resumen_a[(resumen_a["filas"] != 25) | (resumen_a["ultima_hora"] < HORA_CIERRE)]
raras = raras.rename(columns={"filas": "Filas", "ultima_hora": "Última lectura"})
display(tabla_simple(formatear_fechas(raras), precision=0))

,Filas,Última lectura
DIA,,
2020-08-05,24,23:59:00
2020-09-30,50,23:59:00
2020-10-15,24,23:59:00
2020-11-30,50,23:59:00
2021-02-28,100,23:59:00
2021-04-30,50,23:59:00
2021-06-30,51,23:59:59
2021-07-13,23,23:00:00
2021-09-30,50,23:59:00


**Hallazgo.** De las 612 fechas del primer archivo, 596 tienen las 25 lecturas esperadas. Las otras se reparten así: 5 con 24 lecturas, 2 con 23 y 1 con solo 8, y **8 con más de 25** (50, 51 o 100), que rompen la regla de la consigna y se analizan en la sección 6.

En 601 fechas la última lectura es las 23:59:00; en 8 es 23:59:01 o 23:59:59 (el mismo cierre con unos segundos de diferencia); en 2 es las 23:00 y en 1 las 07:00. Además, cada hoja termina con filas vacías (479, 134 y 2 según el archivo), que se descartan.

Las lecturas que faltan **a mitad del día** no afectan al consumo diario, porque el valor es acumulado: solo importa que exista la lectura de cierre.

## 6. Fechas con más de 25 filas

**Pregunta:** ¿por qué hay fechas con 50, 51 o 100 filas? ¿Son copias, errores o días distintos?

Un **bloque** es un día completo de lecturas, de 00:00 a 23:59. Un bloque nuevo empieza cada vez que la hora retrocede respecto de la fila anterior (`numerar_bloques`); se usa la hora y no "cada 25 filas" porque hay días con lecturas de más o de menos.

In [8]:
cierres = {nombre: cierres_por_bloque(tabla, nombre) for nombre, tabla in tablas.items()}
bloques_por_fecha = cierres[nombres[0]].groupby("DIA").size()

repetidas = resumen_a[resumen_a["filas"] > 25].copy()
repetidas["bloques"] = bloques_por_fecha[repetidas.index]
repetidas["es_fin_de_mes"] = pd.Series(repetidas.index.is_month_end, index=repetidas.index).map({True: "Sí", False: "No"})
repetidas = repetidas.rename(columns={"filas": "Filas", "ultima_hora": "Última lectura", "bloques": "Bloques", "es_fin_de_mes": "¿Fin de mes?"})
display(tabla_simple(formatear_fechas(repetidas), precision=0))

,Filas,Última lectura,Bloques,¿Fin de mes?
DIA,,,,
2020-09-30,50,23:59:00,2,Sí
2020-11-30,50,23:59:00,2,Sí
2021-02-28,100,23:59:00,4,Sí
2021-04-30,50,23:59:00,2,Sí
2021-06-30,51,23:59:59,2,Sí
2021-09-30,50,23:59:00,2,Sí
2021-11-30,50,23:59:00,2,Sí
2022-02-28,100,23:59:00,4,Sí


Un caso concreto: el 30/09/2020 (50 filas). Se muestran la primera lectura, la de las 23:00 y la de cierre de cada bloque.

In [9]:
dia = tabla_a[tabla_a["DIA"] == "2020-09-30"]
numero = numerar_bloques(tabla_a)[dia.index]
dia = dia.assign(Bloque=numero - numero.min())

ejemplo_repetido = dia.loc[dia["HORA"].isin(["00:00:00", "23:00:00", "23:59:00"]), ["DIA", "Bloque", "HORA", COLUMNA_CONSUMO]]
ejemplo_repetido.index.name = "Fila del archivo"
display(tabla_simple(formatear_fechas(ejemplo_repetido), precision=0))

,DIA,Bloque,HORA,Frio (Kw)
Fila del archivo,,,,
2199,2020-09-30,0,00:00:00,0
2222,2020-09-30,0,23:00:00,28609
2223,2020-09-30,0,23:59:00,29623
2224,2020-09-30,1,00:00:00,0
2247,2020-09-30,1,23:00:00,24829
2248,2020-09-30,1,23:59:00,25558


**Hallazgo.** Las 8 fechas con más de 25 filas son **todas fin de mes de un mes corto** (28/02, 30/04, 30/06, 30/09 y 30/11). Las de 50 filas tienen 2 bloques; las de 100, febrero, tienen 4, como si febrero tuviera 31 días; el 30/06/2021 tiene 2 bloques más una lectura suelta (51 filas).

No son copias: el 30/09/2020 contiene dos días completos, cada uno con su reinicio a 0 a las 00:00 y su propio cierre (29.623 y 25.558). Esto sugiere que para esas fechas el archivo trae más de un día; las secciones 8 y 9 lo verifican.

## 7. Los tres archivos juntos: períodos, solapes y diferencias

**Pregunta:** ¿qué período cubre cada Excel, cuánto se solapan y, en los días que están en dos, ¿dicen lo mismo?

La consigna pide cuantificar las fechas repetidas entre archivos y definir una regla de desempate. Como cada bloque puede repetirse, se lo identifica con la pareja **(fecha rotulada, número de bloque dentro de esa fecha)**.

In [10]:
resumen_archivos = pd.DataFrame({
    "Primera fecha": {n: c["DIA"].min() for n, c in cierres.items()},
    "Última fecha": {n: c["DIA"].max() for n, c in cierres.items()},
    "Fechas distintas": {n: c["DIA"].nunique() for n, c in cierres.items()},
    "Bloques diarios": {n: len(c) for n, c in cierres.items()},
})

fechas = {n: set(c["DIA"]) for n, c in cierres.items()}
pares = [(nombres[0], nombres[1]), (nombres[1], nombres[2]), (nombres[0], nombres[2])]
solapes = pd.DataFrame({"Fechas en común": {f"{a} y {b}": len(fechas[a] & fechas[b]) for a, b in pares}})

mostrar_lado_a_lado_simple(
    [("Período que cubre cada archivo", formatear_fechas(resumen_archivos)), ("Solape entre archivos", solapes)],
    precision=0,
)

Ahora se compara, para cada bloque que está en dos archivos, el consumo de cierre de la versión más antigua con el de la más reciente.

In [11]:
todos = pd.concat(cierres.values(), ignore_index=True)
claves = ["DIA", "bloque"]

repetidos = todos[todos.duplicated(claves, keep=False)].sort_values(claves + ["archivo"])
antigua = repetidos.drop_duplicates(claves, keep="first").set_index(claves)
reciente = repetidos.drop_duplicates(claves, keep="last").set_index(claves)

comparacion = pd.DataFrame({
    "Archivo antiguo": antigua["archivo"],
    "Consumo antiguo": antigua[COLUMNA_CONSUMO],
    "Archivo reciente": reciente["archivo"],
    "Consumo reciente": reciente[COLUMNA_CONSUMO],
    "ambos_completos": antigua["completo"] & reciente["completo"],
})
comparacion["Diferencia"] = comparacion["Consumo antiguo"] - comparacion["Consumo reciente"]

par = comparacion["Archivo antiguo"] + " y " + comparacion["Archivo reciente"]
son_distintos = comparacion["ambos_completos"] & (comparacion["Diferencia"] != 0)
resumen_comparacion = pd.DataFrame({
    "Bloques en común": par.value_counts(),
    "Completos en ambos": par[comparacion["ambos_completos"]].value_counts(),
    "Con distinto consumo de cierre": par[son_distintos].value_counts(),
}).fillna(0).astype(int)

# Las tres columnas cuentan bloques, así que comparten escala (axis=None).
mostrar_lado_a_lado(
    [("Bloques que están en dos archivos, por pareja de archivos", resumen_comparacion, {"axis": None, "vmin": 0})],
    precision=0,
)

,Bloques en común,Completos en ambos,Con distinto consumo de cierre
2020-2022 y 2021-2023,259,256,5
2021-2023 y 2022-2023,249,247,0


Los bloques con distinto consumo de cierre:

In [12]:
discrepancias = comparacion[son_distintos].drop(columns="ambos_completos").reset_index()
display(tabla_simple(formatear_fechas(discrepancias), precision=0))

,DIA,bloque,Archivo antiguo,Consumo antiguo,Archivo reciente,Consumo reciente,Diferencia
0,2021-07-28,0,2020-2022,2900770,2021-2023,30764,2870006
1,2021-08-25,0,2020-2022,2907122,2021-2023,37122,2870000
2,2021-11-28,0,2020-2022,6155928,2021-2023,25719,6130209
3,2022-02-02,0,2020-2022,52655,2021-2023,19887,32768
4,2022-02-18,0,2020-2022,8861470,2021-2023,21470,8840000


**Hallazgo.** Los tres períodos se encadenan sin huecos de cobertura: el primero llega al 16/03/2022, el segundo cubre del 01/07/2021 al 06/03/2023 y el tercero del 01/07/2022 al 26/10/2023. El primero y el segundo comparten 259 bloques; el segundo y el tercero, 249; el primero y el tercero no se tocan.

En los bloques que están en dos archivos, **el consumo de cierre coincide en todos los casos menos 5**, todos entre el primer y el segundo Excel. Los bloques completos en ambos son 256 de 259 y 247 de 249; el resto está cortado o sin lectura de las 23:59.

En esas 5 el primer archivo trae un error: en cuatro, un consumo del orden de millones (diferencias de 2.870.000, 2.870.006 y 8.840.000, casi redondas, y 6.130.209 el 28/11/2021); en la quinta, el 02/02/2022, 52.655 contra 19.887, una diferencia de exactamente **32.768** (2 elevado a la 15, cifra típica de errores informáticos, no de consumo). En las 5 el valor del segundo archivo está dentro del rango normal.

**D3:** ante dos versiones se elige la completa y plausible, y si empatan la del archivo más reciente.

## 8. El calendario: fechas ausentes y bloques por mes

**Pregunta:** ¿las fechas del archivo coinciden con el calendario? ¿Falta algún día?

Primero, qué fechas del calendario no aparecen en cada archivo dentro de su propio período. Después, cuántos bloques tiene cada mes comparado con los días que realmente tiene, en los dos años completos que cubren los datos (2021 y 2022).

In [13]:
ausentes = {}
for nombre, cierre in cierres.items():
    calendario = pd.date_range(cierre["DIA"].min(), cierre["DIA"].max())
    ausentes[nombre] = calendario.difference(pd.DatetimeIndex(cierre["DIA"].unique()))

fechas_ausentes = pd.DataFrame({
    "Cantidad": {n: len(a) for n, a in ausentes.items()},
    "Fechas ausentes": {n: ", ".join(a.strftime("%d/%m/%Y")) for n, a in ausentes.items()},
})
display(tabla_simple(fechas_ausentes, precision=0))

,Cantidad,Fechas ausentes
2020-2022,12,"29/08/2020, 30/08/2020, 31/08/2020, 31/10/2020, 31/12/2020, 31/03/2021, 31/05/2021, 29/08/2021, 30/08/2021, 31/08/2021, 31/10/2021, 31/12/2021"
2021-2023,12,"29/08/2021, 30/08/2021, 31/08/2021, 31/10/2021, 31/12/2021, 31/03/2022, 31/05/2022, 29/08/2022, 30/08/2022, 31/08/2022, 31/10/2022, 31/12/2022"
2022-2023,9,"29/08/2022, 30/08/2022, 31/08/2022, 31/10/2022, 31/12/2022, 31/03/2023, 31/05/2023, 30/08/2023, 31/08/2023"


In [14]:
unicos = todos.drop_duplicates(["DIA", "bloque"])
bloques_por_mes = unicos.groupby([unicos["DIA"].dt.year, unicos["DIA"].dt.month]).size().unstack(0)

meses = ["Enero", "Febrero", "Marzo", "Abril", "Mayo", "Junio", "Julio", "Agosto", "Septiembre", "Octubre", "Noviembre", "Diciembre"]
tabla_meses = pd.DataFrame(index=range(1, 13))
tabla_meses["Días del mes"] = [pd.Period(f"2021-{mes:02d}").days_in_month for mes in range(1, 13)]
for anio in (2021, 2022):
    tabla_meses[f"Bloques {anio}"] = bloques_por_mes[anio].astype(int)
    tabla_meses[f"Diferencia {anio}"] = tabla_meses[f"Bloques {anio}"] - tabla_meses["Días del mes"]
tabla_meses.index = meses

columnas_bloques = ["Días del mes", "Bloques 2021", "Bloques 2022"]
columnas_diferencia = ["Diferencia 2021", "Diferencia 2022"]
suma_del_anio = tabla_meses.sum().to_frame("Suma del año").T

# El color va solo donde dice algo: las cantidades (28 a 31) se leen mejor como números y
# un gradiente estiraría una diferencia de tres días hasta parecer enorme. Las diferencias
# sí llevan una paleta divergente centrada en 0 (azul = bloques de menos, rojo = de más).
mostrar_lado_a_lado(
    [
        ("Días del mes y bloques del archivo", tabla_meses[columnas_bloques], {"sin_color": True}),
        ("Diferencia (bloques - días reales)", tabla_meses[columnas_diferencia], {"axis": None, "cmap": "coolwarm", "vmin": -3, "vmax": 3}),
        ("Suma de cada columna en el año", suma_del_anio, {"sin_color": True}),
    ],
    precision=0,
)

,Días del mes,Bloques 2021,Bloques 2022
Enero,31,31,31
Febrero,28,31,31
Marzo,31,30,30
Abril,30,31,31
Mayo,31,30,30
Junio,30,31,31
Julio,31,31,31
Agosto,31,28,28
Septiembre,30,31,31
Octubre,31,30,30


**Hallazgo.** Las fechas ausentes se repiten igual cada año: el 31 de marzo, mayo, octubre y diciembre, y los días 29, 30 y 31 de agosto (en el tercer archivo, en 2023, faltan solo el 30 y el 31 de agosto). Al revés, los meses cortos tienen bloques de más: febrero tiene 31 en lugar de 28, y abril, junio, septiembre y noviembre tienen 31 en lugar de 30.

**Los desvíos se compensan:** la suma de las diferencias da 0 y cada año (2021 y 2022) tiene exactamente 365 bloques, igual que sus 365 días. Una compensación exacta como esa es difícil de atribuir al azar. La lectura más razonable, aunque **no demostrada**, es que los bloques son días consecutivos reales y que las etiquetas de fecha están desfasadas unos días en algunos meses.

## 9. Evidencia de la hoja `Seguimiento Dia`

**Pregunta:** ¿hay en el propio Excel alguna señal de cómo trata la planta a los bloques que repiten fecha?

El segundo archivo (y solo ese) trae una hoja extra, `Seguimiento Dia`, con una columna `Ultimo Dato del Dia` que vale 1 en la última lectura de cada día y 0 en el resto. Se la compara con los bloques que encontramos.

In [15]:
seguimiento = pd.read_excel(libros[nombres[1]], sheet_name="Seguimiento Dia")
marcas = seguimiento[seguimiento["Ultimo Dato del Dia"] == 1]
bloques_b = cierres[nombres[1]]

evidencia = pd.DataFrame({"Cantidad": {
    "Filas de la hoja Seguimiento Dia": len(seguimiento),
    "Filas de Consolidado EE (sin las vacías)": len(tablas[nombres[1]]),
    "Marcas de 'Ultimo Dato del Dia'": len(marcas),
    "Bloques diarios del segundo archivo": len(bloques_b),
    "Fechas distintas del segundo archivo": bloques_b["DIA"].nunique(),
    "Marcas en una hora anterior a las 23:59": int((marcas["HORA"].astype(str) < HORA_CIERRE).sum()),
}})
display(tabla_simple(evidencia, precision=0))

,Cantidad
Filas de la hoja Seguimiento Dia,15316
Filas de Consolidado EE (sin las vacías),15316
Marcas de 'Ultimo Dato del Dia',614
Bloques diarios del segundo archivo,614
Fechas distintas del segundo archivo,602
Marcas en una hora anterior a las 23:59,4


**Hallazgo.** La hoja `Seguimiento Dia` tiene exactamente las mismas filas que `Consolidado EE` del segundo archivo (15.316) y **614 marcas** de último dato, igual a los 614 bloques y no a las 602 fechas distintas: para la planta, cada bloque cuenta como un día. Cuatro de esas marcas caen antes de las 23:59, o sea que la hoja marca el final de **cada bloque**, también el de los días cortados.

Hay que tomarlo como una pista y no como una prueba independiente: tiene las mismas filas que `Consolidado EE`, probablemente se calcula a partir de ellas, y no trae ninguna fecha real distinta.

## 10. Decisiones sobre los bloques y los días inválidos

**D2. Los bloques son días distintos.** Cada bloque de 00:00 a 23:59 se trata como un día, y "día siguiente" es el bloque siguiente del archivo, no la fecha siguiente del calendario. Lo respaldan: los bloques repetidos tienen valores propios y coinciden entre archivos (sección 7), el total de bloques por año es igual a los días del año (sección 8) y la hoja `Seguimiento Dia` marca cada bloque como un día (sección 9). **Lo que no sabemos** es qué fecha real le corresponde a cada bloque; las etiquetas de fecha pueden estar corridas unos días. Por eso la fecha rotulada se conserva solo como dato informativo y las variables de calendario (día de la semana, mes) hay que usarlas con cuidado. Queda como consulta para la cátedra.

| Alternativa | Por qué se descartó |
|---|---|
| Quedarse con el primer bloque de cada fecha y tirar los demás | Descarta días reales (unos 7 por año) y no corrige las fechas corridas. |
| Reasignar fechas consecutivas según el orden del archivo | Obliga a inventar fechas sin poder comprobarlas. |

**D4. Días inválidos.** Un día es **inválido** si no tiene lectura de las 23:59 (el acumulado del día queda incompleto) o si su consumo supera el corte provisorio de `CONSUMO_MAXIMO_PLAUSIBLE` (errores del orden de millones). No se imputa el valor: se marca `valido = False` y el consumo queda vacío. Imputar el objetivo equivaldría a medir el error del modelo contra un número fabricado por nosotros. Qué hacer con esas filas al entrenar se decide en preprocesamiento.

**D3. Versión elegida.** Si un bloque está en dos Excel, se toma la versión que llega a las 23:59 y tiene un consumo plausible; si empatan, la del archivo más reciente (en las discrepancias de la sección 7 el posterior tenía siempre el valor correcto).

## 11. La tabla `diario`: una versión por bloque

**Pregunta:** aplicando D3, ¿cuántos días únicos quedan y de qué archivo sale cada uno?

In [16]:
mejores = unir_versiones(todos)

versiones = todos.groupby(["DIA", "bloque"]).size().value_counts().sort_index().rename_axis("Archivos que lo traen").to_frame("Cantidad de bloques")
elegidos = mejores["archivo"].value_counts().sort_index().rename_axis("Archivo").to_frame("Bloques elegidos")

mostrar_lado_a_lado(
    [
        ("Cuántos archivos traen cada bloque", versiones, {"vmin": 0}),
        ("De qué archivo se tomó cada bloque", elegidos, {"vmin": 0}),
    ],
    precision=0,
)
print("Bloques únicos (días de la serie):", len(mejores))

,Cantidad de bloques
Archivos que lo traen,
1,704
2,508
,Bloques elegidos
Archivo,
2020-2022,365
2021-2023,365
2022-2023,482


Bloques únicos (días de la serie): 1212


Los días inválidos según D4 (sin lectura de las 23:59 o con consumo implausible), y la comprobación del 02/02/2022, el caso en que el primer archivo tenía 52.655 y el segundo 19.887:

In [17]:
invalidos = mejores.loc[~mejores["completo"] | ~mejores["plausible"], ["DIA", "bloque", "HORA", COLUMNA_CONSUMO, "filas", "archivo"]]
invalidos = invalidos.rename(columns={"bloque": "Bloque", "HORA": "Última lectura", "filas": "Lecturas", "archivo": "Archivo elegido"})
display(tabla_simple(formatear_fechas(invalidos.reset_index(drop=True)), precision=0))

control = mejores.loc[mejores["DIA"] == "2022-02-02", ["DIA", "bloque", COLUMNA_CONSUMO, "archivo"]]
display(tabla_simple(formatear_fechas(control.reset_index(drop=True)), precision=0))

,DIA,Bloque,Última lectura,Frio (Kw),Lecturas,Archivo elegido
0,2021-04-30,1,23:59:00,5572801,25,2020-2022
1,2021-07-13,0,23:00:00,33704,23,2021-2023
2,2022-03-02,0,23:00:00,19294,24,2021-2023
3,2023-02-28,0,23:00:00,15279,24,2022-2023
4,2023-04-13,0,19:00:00,14315,20,2022-2023
5,2023-04-19,0,16:00:00,13439,17,2022-2023
6,2023-10-26,0,07:00:00,3124,8,2022-2023


,DIA,bloque,Frio (Kw),archivo
0,2022-02-02,0,19887,2021-2023


**Hallazgo.** La serie tiene **1.212 días únicos** (bloques): 704 están en un solo archivo y 508 en dos. Como ante un empate gana el archivo más reciente, el tercero aporta todos sus bloques (482), el segundo los que el tercero no trae (365) y el primero los que el segundo no trae (365).

Los **7 días inválidos** (D4) son: el segundo bloque del 30/04/2021, con un consumo de 5,57 millones que está solo en el primer archivo (no hay otra versión que lo corrija), y seis días sin lectura de las 23:59 (13/07/2021, 02/03/2022, 28/02/2023, 13/04/2023, 19/04/2023 y 26/10/2023, este último el último día del dataset, cortado a las 07:00).

La comprobación del 02/02/2022 confirma D3: el bloque sale del segundo archivo con 19.887, no con el 52.655 del primero.

## 12. El objetivo: el consumo del día siguiente

**Pregunta:** ¿cómo se arma la variable a predecir?

El modelo trabaja a las 23:59 de cada día: con lo que ya se sabe de hoy, predice cuánto va a consumir el frío **mañana**. Una fila de la tabla es "lo conocido al cierre del día D" y su objetivo es el consumo del día D+1. Predecir el de hoy con datos de hoy sería predecir algo que ya se conoce (fuga de información).

`construir_diario` agrega tres columnas: `valido` (D4), `consumo_dia` (el consumo de cierre, vacío si el día es inválido) y `objetivo` (`consumo_dia` corrido una fila hacia arriba con `shift(-1)`).

In [18]:
diario = construir_diario(mejores)

conteo = pd.DataFrame({"Cantidad": {
    "Días totales (bloques)": len(diario),
    "Días válidos": int(diario["valido"].sum()),
    "Días inválidos": int((~diario["valido"]).sum()),
    "Filas con objetivo": int(diario["objetivo"].notna().sum()),
    "Filas sin objetivo": int(diario["objetivo"].isna().sum()),
}})

muestra = diario.loc[:5, ["DIA", "bloque", "consumo_dia", "objetivo"]]
sin_objetivo = diario.loc[diario["objetivo"].isna(), ["DIA", "bloque", "consumo_dia", "valido"]]

mostrar_lado_a_lado_simple(
    [
        ("Resumen", conteo),
        ("Las primeras filas: el objetivo es el consumo de la fila siguiente", formatear_fechas(muestra)),
        ("Filas sin objetivo", formatear_fechas(sin_objetivo)),
    ],
    precision=0,
)

**Hallazgo.** De los 1.212 días, 1.205 son válidos y 7 inválidos. Hay **1.204 filas con objetivo** y 8 sin él: los 7 días que preceden a un día inválido (su "mañana" no es confiable) y el último día del dataset (inválido y sin día siguiente).

Esas 8 filas se conservan en la tabla, con su consumo propio, porque más adelante pueden servir de historia para otros días (valores de días anteriores como variables de entrada); recién se descartan al entrenar. En la muestra se ve la mecánica: el objetivo de cada fila es el `consumo_dia` de la siguiente.

## 13. Pendientes

**Valores extremos que pasan el corte provisorio.** El corte de `CONSUMO_MAXIMO_PLAUSIBLE` solo separa los errores de millones. Entre los días válidos hay consumos muy altos y muy bajos que habrá que revisar en el análisis de atípicos: el máximo es 56.128, el mínimo 2.310, y los cinco días más altos caen entre abril y agosto de 2021.

In [23]:
resumen_consumo = diario["consumo_dia"].describe(percentiles=[0.05, 0.5, 0.95]).to_frame("consumo_dia")
resumen_consumo.index = ["días", "promedio", "desvío", "mínimo", "percentil 5", "mediana", "percentil 95", "máximo"]
mas_altos = diario.nlargest(5, "consumo_dia")[["DIA", "bloque", "consumo_dia"]]

mostrar_lado_a_lado_simple(
    [("Consumo de cierre de los días válidos", resumen_consumo), ("Los 5 días válidos con mayor consumo", formatear_fechas(mas_altos))],
    precision=0,
)

NameError: name 'mostrar_lado_a_lado_simple' is not defined

| Pendiente | Detalle |
|---|---|
| Confirmar dudas con profes | Fechas repetidas a fin de mes y bloques por mes que no coinciden con los días reales; nombres de los Excel distintos a los de la consigna; formato del informe; si se aceptan `tests/` y `docs/`. |
| Registros de decisión | Pasar D1 a D4 a `docs/decisions/`, un archivo por decisión. |
| Huella de los Excel | `data/checksums.json` con SHA-256 por bloques. |
| Efecto del día de la semana | Medirlo con un número. Ojo: las etiquetas de fecha pueden estar corridas (D2), y la consigna pide explicar qué pudo borrar el patrón semanal. |
| Atípicos | Revisar los consumos altos de la tabla de arriba y fijar la regla definitiva en lugar del corte provisorio. |
| Otras hojas | Inventario hoja por hoja y alineación con `diario` por la clave (fecha rotulada, bloque); correlaciones y variables que se parecen demasiado al objetivo. |
| Análisis temporal y gráficos | Evolución, estacionalidad, histograma y Q-Q del objetivo, boxplots mensuales. |

Los roles son los del plan: **A** Datos y Pipeline, **B** Análisis e Informe, **C** Modelado y MLOps.